<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_28_concurrency_intro/note_lesson_28_concurrency.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Урок 28 — Потоки, процеси, asyncio: ранковий звіт диспетчерської

Диспетчерська «Смачно + Таксі» щоранку робить дві речі:

1. **опитує 5 ресторанів**, чи вони відкриті (кожна відповідь — 0,5 с очікування мережі);
2. **рахує важку статистику** по 4 районах (процесор зайнятий на 100 %).

Послідовно це займає кілька секунд. Сьогодні пришвидшуємо обидві задачі трьома інструментами — і вчимося обирати, який із них брати.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | задачі, що **чекають**, і задачі, що **рахують** | 0.1–0.2 | |
| 1 | **потоки**: `ThreadPoolExecutor`, `max_workers`, `submit` і `as_completed`, винятки, черга `queue.Queue` | 1.1–1.7 | |
| 2 | **стан гонитви**: чому `+=` не атомарний, `Lock`, «перевірив — і запізно», взаємне блокування | 2.1–2.6 | **Задача 1** |
| 3 | **GIL і процеси**: `ProcessPoolExecutor`, окрема пам'ять, `pickle`, ціна процесів, конвеєр «потоки + процеси» | 3.1–3.8 | **Задача 2** |
| 4 | **`asyncio`**: корутини, `await`, `gather`, `create_task`, `as_completed`, помилки, `TaskGroup`, тайм-аути, черга | 4.1–4.13 | |
| 5 | блокуючий код в `asyncio`: `to_thread` | 5.1–5.2 | |
| 6 | обмежити одночасність: `Semaphore` | 6.1 | **Задача 3** |
| 7 | одна задача — чотири способи; що обрати | 7.1 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи) лише створює функцію — **запуск завжди в наступній клітинці**. Так можна перезапускати приклад, не перевизначаючи функції.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Час у тебе буде трохи іншим, ніж у тексті, — важливе **співвідношення**: «у 5 разів швидше», а не «0,5 с».
- Теорія й покрокові схеми — у книзі: [Урок 28](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_28/).

---
## 0. Чекати чи рахувати

Функції для **процесів** мають жити у **файлі**: на Windows і macOS процес-працівник імпортує їх заново, а клітинку ноутбука імпортувати не можна. Тому запишемо їх у модуль `district_stats.py` — клітинка з `%%writefile` створює файл поруч з ноутбуком (і в Jupyter, і в Colab).

In [ ]:
%%writefile district_stats.py
"""Функції, які запускаємо в процесах: мають бути у файлі, щоб процеси могли їх імпортувати."""
import os
import time

orders = []


def count_primes(limit):
    """Кількість простих чисел, менших за limit. Навмисно повільно: перебір дільників."""
    count = 0
    for n in range(2, limit):
        for d in range(2, int(n ** 0.5) + 1):
            if n % d == 0:
                break
        else:
            count += 1
    return count


def add_order(name):
    orders.append(name)      # додає у СВОЮ копію списку
    return len(orders)


def worker_pid(task):
    """Повертає номер процесу, який виконав задачу."""
    time.sleep(0.05)         # трохи роботи, щоб задачі розійшлися між працівниками
    return os.getpid()

Тепер — імпорти й дві «моделі» задач:

- `check_restaurant(name)` — опитати ресторан: **чекає** 0,5 с (як запит у мережу);
- `district_stats.count_primes(limit)` — порахувати статистику району: **рахує**.

І допоміжна `timed(label, work)`: виконує `work()` і друкує, скільки це зайняло.

In [ ]:
import asyncio
import queue
import sys
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed

import district_stats

RESTAURANTS = ["Борщ і Ко", "Піца Поділ", "Суші Оболонь", "Вареники 24/7", "Шаурма Центр"]
DISTRICTS = [150_000] * 4


def check_restaurant(name):
    time.sleep(0.5)            # чекаємо відповідь мережі
    return f"{name}: відкрито"


def timed(label, work):
    start = time.perf_counter()
    result = work()
    print(f"{label}: {time.perf_counter() - start:.1f} с")
    return result

### Приклад 0.1. Скільки це займає без жодних хитрощів

Спершу — **вимір**. Без нього не зрозуміти, чи щось пришвидшилось.

In [ ]:
timed("5 ресторанів послідовно", lambda: [check_restaurant(n) for n in RESTAURANTS])
stats = timed("4 райони послідовно", lambda: [district_stats.count_primes(n) for n in DISTRICTS])

5 × 0,5 = 2,5 с на ресторани. Але ці дві задачі **різні за природою**:

- **Ресторани** — задача, що **чекає** (I/O-bound): процесор простоює, поки йде відповідь мережі. За ці 2,5 с процесор майже нічого не робив.
- **Райони** — задача, що **рахує** (CPU-bound): процесор зайнятий на 100 % весь час.

Два слова, які далі знадобляться:

- **Конкурентність** — кілька задач просуваються за той самий проміжок часу, але не обов'язково в ту саму мить. П'ять ресторанів, які ми чекаємо разом, — конкурентність.
- **Паралелізм** — робота виконується буквально одночасно, наприклад на кількох ядрах процесора.

| Тип задачі | Що допомагає | Чому |
|---|---|---|
| **чекає** (мережа, диск, база) | потоки або `asyncio` | поки одна задача чекає, інші працюють |
| **рахує** (цикли, формули на чистому Python) | процеси | кожен процес — на своєму ядрі |

### Приклад 0.2. Чекає чи рахує?

Ось як класифікувати типові задачі диспетчерської. Головне питання: **на що йде час — на очікування чи на обчислення?**

In [ ]:
kinds = {
    "завантажити 100 сторінок меню з сайтів ресторанів": "IO",   # час іде на мережу
    "стиснути 500 фото страв": "CPU",                            # процесор перераховує пікселі
    "прочитати 20 файлів звітів з диска": "IO",                  # чекаємо диск
    "порахувати найкоротші маршрути для 10 000 поїздок": "CPU",  # алгоритм на графі
    "надіслати 1000 SMS через API": "IO",                        # 1000 запитів у мережу
}

In [ ]:
for task, kind in kinds.items():
    tool = "потоки / asyncio" if kind == "IO" else "процеси"
    print(f"{kind:3} → {tool:16} | {task}")

---
## 1. Потоки: чекати одночасно

**Потік** — окрема лінія виконання всередині однієї програми (процесу). Потоки однієї програми ділять пам'ять: бачать ті самі змінні. Потоки допомагають ресторанам, бо під час `time.sleep` чи мережевого запиту потік **відпускає GIL** (розділ 3) — і поки він чекає, працюють інші.

Найпростіший спосіб — **пул потоків** `ThreadPoolExecutor`: він сам створює потоки, роздає їм задачі й збирає результати.

**🔮 Прогноз:** скільки займе опитування 5 ресторанів через `ThreadPoolExecutor`? У якому порядку будуть результати?

<details>
<summary>Відповідь</summary>

Близько 0,5 с: п'ять очікувань ідуть одночасно. `pool.map` повертає результати **в порядку вхідного списку**, хоч би в якому порядку закінчились потоки.

</details>

### Приклад 1.1. `pool.map` — те саме, що `map`, але в потоках

In [ ]:
with ThreadPoolExecutor() as pool:
    results = timed("5 ресторанів у потоках", lambda: list(pool.map(check_restaurant, RESTAURANTS)))
print(results)

Що сталося рядок за рядком:

- `with ThreadPoolExecutor() as pool:` — створили пул; на виході з `with` він **дочекається** всіх потоків і прибере їх;
- `pool.map(check_restaurant, RESTAURANTS)` — роздав по ресторану кожному потоку; повертає **ліниві** результати, тому загортаємо в `list(...)`;
- результати — у порядку `RESTAURANTS`.

### Приклад 1.2. `check_all(names)` — функція, яку можна перевикористати

Загорнемо опитування у функцію. Визначення — в одній клітинці, виклик і перевірка — в наступній.

In [ ]:
def check_all(names):
    """Опитати всі ресторани в потоках; порядок — як у names."""
    with ThreadPoolExecutor() as pool:
        return list(pool.map(check_restaurant, names))

In [ ]:
start = time.perf_counter()
statuses = check_all(RESTAURANTS)
elapsed = time.perf_counter() - start
print(statuses)
print(f"{elapsed:.1f} с")
assert statuses == [f"{n}: відкрито" for n in RESTAURANTS]
assert elapsed < 1.5, "схоже, ресторани опитуються по черзі"

**🔮 Прогноз:** скільки займе опитування, якщо обмежити пул двома потоками — `max_workers=2`?

<details>
<summary>Відповідь</summary>

Близько 1,5 с. Одночасно працюють лише два потоки: ресторани йдуть «хвилями» 2 + 2 + 1, кожна по 0,5 с. Без `max_workers` пул бере `min(32, кількість ядер + 4)` потоків — тому вище всі п'ять ішли разом. Для порівнянь задавай `max_workers` явно, щоб результат не залежав від машини.

</details>

### Приклад 1.3. `max_workers` — скільки потоків одночасно

In [ ]:
for workers in (1, 2, 5):
    with ThreadPoolExecutor(max_workers=workers) as pool:
        timed(f"max_workers={workers}", lambda: list(pool.map(check_restaurant, RESTAURANTS)))

`max_workers=1` — це знову послідовно (2,5 с). 2 потоки — 3 хвилі (1,5 с). 5 потоків — одна хвиля (0,5 с). Більше потоків, ніж задач, нічого не дасть.

### Приклад 1.4. Хто що робить: журнал потоків

Подивимось усередину: який потік узяв який ресторан і коли. `threading.current_thread().name` — ім'я поточного потоку. Записи складаємо в список і друкуємо **після** роботи: якщо друкувати з кількох потоків одночасно, рядки можуть перемішатися.

In [ ]:
journal = []


def check_logged(name):
    thread = threading.current_thread().name
    journal.append((time.perf_counter() - start, thread, "почав   ", name))
    result = check_restaurant(name)
    journal.append((time.perf_counter() - start, thread, "закінчив", name))
    return result

In [ ]:
journal.clear()
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=2, thread_name_prefix="диспетчер") as pool:
    list(pool.map(check_logged, RESTAURANTS))

for moment, thread, action, name in sorted(journal):
    print(f"{moment:4.2f} с  {thread:<12} {action} {name}")

Видно хвилі: у момент 0,00 два потоки почали два ресторани; приблизно о 0,50 обидва закінчили й одразу взяли наступні. Потоки **перевикористовуються**: пул не створює новий потік на кожну задачу.

### Приклад 1.5. `submit` і `as_completed`: результат — щойно готовий

`pool.map` повертає результати в порядку входу: якщо перший ресторан найповільніший, решта чекає на нього. Інший спосіб — `pool.submit(функція, аргументи)`: він одразу повертає **`Future`** («квитанцію» на майбутній результат). `as_completed(futures)` видає квитанції в порядку **готовності**.

In [ ]:
DELAYS = {"Борщ і Ко": 0.6, "Піца Поділ": 0.2, "Суші Оболонь": 0.4}


def check_with_pause(name, delay):
    time.sleep(delay)
    return f"{name}: відкрито (відповів за {delay} с)"

In [ ]:
start = time.perf_counter()
with ThreadPoolExecutor() as pool:
    futures = [pool.submit(check_with_pause, name, delay) for name, delay in DELAYS.items()]
    for future in as_completed(futures):
        print(f"{time.perf_counter() - start:.1f} с  {future.result()}")

| | `pool.map(f, items)` | `pool.submit(f, x)` + `as_completed` |
|---|---|---|
| Порядок результатів | як у вхідному списку | як закінчились |
| Аргументи | один список | будь-які, різні для кожної задачі |
| Коли брати | треба всі результати разом | показувати прогрес, реагувати на перші відповіді |

### Приклад 1.6. Помилка в потоці

Що, як один ресторан не відповів і функція кинула виняток? **Потік не друкує помилку сам** — вона зберігається у `Future` і вилітає, коли ти просиш результат: `future.result()`.

In [ ]:
def check_maybe_closed(name):
    time.sleep(0.1)
    if name == "Суші Оболонь":
        raise ConnectionError(f"{name}: сервер не відповідає")
    return f"{name}: відкрито"

In [ ]:
with ThreadPoolExecutor() as pool:
    futures = {pool.submit(check_maybe_closed, name): name for name in RESTAURANTS}

for future, name in futures.items():
    try:
        print("ok     ", future.result())
    except ConnectionError as error:
        print("помилка", error)

Один збій не зупинив інших: решта ресторанів опитані. З `pool.map` інакше — виняток вилетить, щойно цикл дійде до збійного результату, і наступні результати ти вже не отримаєш:

In [ ]:
with ThreadPoolExecutor() as pool:
    try:
        for status in pool.map(check_maybe_closed, RESTAURANTS):
            print(status)
    except ConnectionError as error:
        print("map зупинився на:", error)

### Приклад 1.7. Потік вручну і черга `queue.Queue`

Пул — не єдиний спосіб. `threading.Thread(target=функція, args=(...))` створює потік вручну: `start()` запускає, `join()` чекає завершення. Але такий потік **викидає** те, що повертає функція, — результат доведеться складати в спільну змінну самому:

In [ ]:
statuses_by_hand = {}


def check_into_dict(name):
    statuses_by_hand[name] = check_restaurant(name)   # return тут нікому не потрібен

In [ ]:
thread = threading.Thread(target=check_into_dict, args=("Борщ і Ко",))
thread.start()
thread.join()                                          # чекаємо, поки потік закінчить
print(statuses_by_hand)

Ручні потоки зручні для **довгоживучих працівників**, які беруть роботу з **черги**. `queue.Queue` — потокобезпечна черга: `put` кладе, `get` бере (і чекає, якщо порожньо). Кухня кладе готові замовлення, двоє кур'єрів їх розвозять. `None` у черзі — сигнал «зміна закінчилась».

In [ ]:
orders_queue = queue.Queue()
delivered = []


def courier(name):
    while True:
        order = orders_queue.get()       # чекає, поки з'явиться замовлення
        if order is None:                # сигнал: замовлень більше не буде
            break
        time.sleep(0.2)                  # везе
        delivered.append((name, order))

In [ ]:
delivered.clear()
start = time.perf_counter()
couriers = [threading.Thread(target=courier, args=(name,)) for name in ("Тарас", "Марія")]
for worker in couriers:
    worker.start()

for order in ["борщ", "піца", "суші", "вареники", "шаурма", "узвар"]:
    orders_queue.put(order)              # кухня віддає замовлення
for _ in couriers:
    orders_queue.put(None)               # по одному сигналу «стоп» на кур'єра
for worker in couriers:
    worker.join()

print(f"{len(delivered)} доставок за {time.perf_counter() - start:.1f} с")
for name in ("Тарас", "Марія"):
    print(name, "→", [order for who, order in delivered if who == name])

6 доставок по 0,2 с на двох кур'єрів — близько 0,6 с. Хто що взяв, щоразу трохи інакше: черга віддає замовлення тому, хто перший звільнився. Це патерн **«виробник — споживач»** (producer–consumer): він же — в основі черг задач Celery (модуль 4).

`delivered.append(...)` з кількох потоків тут безпечний: `append` у CPython — одна неподільна операція. А от «прочитати, змінити, записати» — вже ні. Про це наступний розділ.

---
## 2. Стан гонитви

**Стан гонитви** (race condition) — результат залежить від того, в якому порядку потоки випадково виконали свої кроки.

### Приклад 2.1. `+=` — це три кроки, а не один

`balance += 1` виглядає однією дією. Подивимось, на що його розкладає Python, — модуль `dis` показує байт-код:

In [ ]:
import dis

balance = 0


def add_one():
    global balance
    balance += 1

In [ ]:
dis.dis(add_one)

Серед рядків є три головні: `LOAD_GLOBAL balance` (**прочитати**), `BINARY_OP +=` (**додати**; у Python 3.10 цей рядок називається `INPLACE_ADD`), `STORE_GLOBAL balance` (**записати**). Між будь-якими двома кроками Python може перемкнутися на інший потік. Тоді два потоки прочитають те саме старе значення — і одна з оплат загубиться.

### Приклад 2.2. Чотири каси, одна змінна

Чотири каси одночасно зараховують по 1000 оплат по 1 грн. `time.sleep(0)` між «прочитати» і «записати» лише віддає чергу іншому потоку — щоб проблема проявилася наочно, а не раз на мільйон.

**🔮 Прогноз:** скільки буде на рахунку після роботи чотирьох кас?

<details>
<summary>Відповідь</summary>

Значно менше за 4000 — щоразу інше число (у наших 30 запусках — від 1000 до 1336). Каси читають старе значення й записують поверх чужих оплат.

</details>

In [ ]:
def pay(times):
    global balance
    for _ in range(times):
        current = balance        # 1. прочитати
        time.sleep(0)            # каса на мить відволіклася
        balance = current + 1    # 2. записати


def run_cashboxes(work, cashboxes=4, payments=1000):
    """Запустити каси в потоках і дочекатися всіх."""
    threads = [threading.Thread(target=work, args=(payments,)) for _ in range(cashboxes)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

In [ ]:
balance = 0
run_cashboxes(pay)
print("Очікували 4000, на рахунку:", balance)

Запусти клітинку кілька разів — число щоразу інше. Саме тому гонитви такі підступні: тест може пройти 99 разів і впасти на сотий.

### Приклад 2.3. `Lock` — по одному

**Замок** (`threading.Lock`) пускає у блок `with lock:` лише **один** потік; решта чекають біля входу. Під замок кладемо **весь** блок «прочитати — записати»:

In [ ]:
lock = threading.Lock()


def pay_safe(times):
    global balance
    for _ in range(times):
        with lock:                   # один потік за раз
            current = balance
            time.sleep(0)
            balance = current + 1

In [ ]:
balance = 0
run_cashboxes(pay_safe)
print("На рахунку:", balance)
assert balance == 4000

`with lock:` — те саме, що `lock.acquire()` на вході і `lock.release()` на виході, навіть якщо всередині стався виняток. Тому завжди пиши саме `with`.

Ціна замка: усередині `with lock:` каси працюють **по черзі**. Чим більше коду під замком, тим менше користі від потоків. Під замок — лише те, що справді чіпає спільні дані.

### Приклад 2.4. Без спільних змінних

Кращий шлях — не ділити змінну взагалі. Кожна каса **повертає**, скільки оплат прийняла, а головна програма підсумовує результати `pool.map`. Жодного `global`, жодного `Lock`.

In [ ]:
def count_payments(times):
    """Каса приймає times оплат по 1 грн і повертає свою суму."""
    total = 0                     # локальна змінна — своя в кожного потоку
    for _ in range(times):
        time.sleep(0)
        total += 1
    return total


def total_payments(cashboxes):
    with ThreadPoolExecutor() as pool:
        return sum(pool.map(count_payments, cashboxes))

In [ ]:
print(total_payments([1000, 1000, 1000, 1000]))
assert total_payments([1000, 1000, 1000, 1000]) == 4000
assert total_payments([5, 7]) == 12
assert total_payments([]) == 0

Правило: **спершу намагайся не ділити дані** (кожен потік працює зі своїм і повертає результат). Замок — коли ділити справді доводиться.

### Приклад 2.5. Перевірив — і вже запізно

Диспетчерська імпортує меню ресторанів і веде список `imported` — хто вже імпортований. Деякі назви приходять у завдання кілька разів. Код перевіряє «чи вже є?» і лише тоді дописує. Між перевіркою й записом — збереження меню в базу (0,01 с).

**🔮 Прогноз:** 8 завдань, але лише 2 різні ресторани. Скільки записів буде в `imported`?

<details>
<summary>Відповідь</summary>

8 — кожна назва чотири рази. Потоки одночасно прокидаються після мережі, усі четверо бачать «ще немає» і всі четверо дописують. Це **check-then-act** («перевір, потім дій») — класичний стан гонитви.

**GIL тут не рятує.** Він не робить операцію з кількох кроків атомарною: між `if` і `append` інший потік встигає своє.

</details>

In [ ]:
TASKS = ["Борщ і Ко", "Піца Поділ"] * 4      # 8 завдань, 2 ресторани


def save_menu(name):
    time.sleep(0.01)                         # пишемо меню в базу


def import_menu(name):
    check_restaurant(name)                   # завантаження: 0,5 с мережі
    if name not in imported:                 # 1. перевірити
        save_menu(name)
        imported.append(name)                # 2. дописати

In [ ]:
imported = []
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(import_menu, TASKS))
print(len(imported), "записів:", imported)

### Задача 1. Перевірка й запис — під одним замком

Допиши `import_menu_safe` повністю. Правила:

- `if name not in imported` і `imported.append(name)` — разом під `with imported_lock:`; між ними — `save_menu(name)`;
- повільне завантаження `check_restaurant(name)` — **поза** замком. Інакше вісім завантажень підуть по черзі (4 с), і потоки втратять сенс.

Перевірка — у наступній клітинці.

In [ ]:
imported_lock = threading.Lock()


def import_menu_safe(name):
    # BEGIN SOLUTION
    check_restaurant(name)                   # повільне — поза замком
    with imported_lock:
        if name not in imported:
            save_menu(name)
            imported.append(name)
    # END SOLUTION

In [ ]:
imported = []
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(import_menu_safe, TASKS))
elapsed = time.perf_counter() - start
print(imported, f"| {elapsed:.1f} с")
assert sorted(imported) == ["Борщ і Ко", "Піца Поділ"], "у списку дублікати: перевірка й запис мають бути під одним замком"
assert elapsed < 1.5, "завантаження опинилось під замком — винеси check_restaurant за with"
print("✅ Задача 1 розв'язана")

Замок прибрав дублікати **в списку**, але «Борщ і Ко» все одно **завантажили** чотири рази. Щоб не качати двічі, треба до завантаження позначати ресторан як «вже в роботі» і окремо вирішувати, що робити, якщо завантаження впало. Це вже інша, складніша задача.

### Приклад 2.6. Взаємне блокування (deadlock)

Два замки — два способи все зламати. Диспетчер А бере замок **кухні**, потім **кур'єрів**. Диспетчер Б — навпаки. Кожен тримає свій і чекає чужий — назавжди. Щоб приклад не завис, другий замок чекаємо з тайм-аутом (`patience`): А терпить 1 с, Б — 3 с. Без тайм-ауту клітинка не закінчилася б ніколи.

In [ ]:
kitchen_lock = threading.Lock()
courier_lock = threading.Lock()


def dispatcher(name, first, second, patience):
    with first:
        time.sleep(0.1)                         # встиг узяти свій замок
        if second.acquire(timeout=patience):    # чекаємо другий не довше patience секунд
            second.release()
            return f"{name}: узяв обидва замки"
        return f"{name}: {patience} с чекав другий замок — взаємне блокування, здаюся"

In [ ]:
with ThreadPoolExecutor() as pool:
    a = pool.submit(dispatcher, "А", kitchen_lock, courier_lock, 1)   # кухня → кур'єри
    b = pool.submit(dispatcher, "Б", courier_lock, kitchen_lock, 3)   # кур'єри → кухня
print(a.result())
print(b.result())

Подивись уважно: А чекав секунду, здався і **відпустив** кухню — лише тоді Б (він терплячіший) отримав обидва замки. Без `timeout` ніхто б не здався: обидва потоки чекали б вічно, а з ними й уся програма.

Взаємне блокування можливе лише тоді, коли одночасно виконуються **чотири умови**: (1) ресурс може тримати лише один (замок); (2) потік тримає один ресурс і чекає інший; (3) ресурс не можна відібрати силоміць; (4) **цикл очікування**: А чекає Б, Б чекає А. Достатньо зламати одну. Найпростіше — четверту: **завжди брати замки в одному порядку**.

In [ ]:
with ThreadPoolExecutor() as pool:
    a = pool.submit(dispatcher, "А", kitchen_lock, courier_lock, 1)
    b = pool.submit(dispatcher, "Б", kitchen_lock, courier_lock, 1)   # той самий порядок
print(a.result())
print(b.result())

---
## 3. GIL і процеси

**GIL** (Global Interpreter Lock) — замок у звичайному CPython: у кожен момент байт-код Python виконує **лише один потік**. Під час очікування (I/O) потік його відпускає — тому потоки й допомогли ресторанам. А під час обчислень на чистому Python потоки лише чергуються. Бібліотеки, написані на C (наприклад, NumPy), можуть відпускати GIL і під час обчислень — там картина інша.

З Python 3.14 є окрема, офіційно підтримувана збірка **без GIL** (free-threaded). Її видно в `python -VV` (рядок містить `free-threading build`). Цей урок — про звичайну збірку з GIL.

### Приклад 3.1. Перевір свою збірку

In [ ]:
print(sys.version)
gil_check = getattr(sys, "_is_gil_enabled", None)      # з'явилась у Python 3.13
print("GIL увімкнено:", gil_check() if gil_check else True)

**🔮 Прогноз:** чи пришвидшать 4 потоки підрахунок `count_primes` для 4 районів? А 4 процеси?

<details>
<summary>Відповідь</summary>

Потоки — ні: через **GIL** байт-код у кожен момент виконує лише один потік, вони чергуються. Процеси — так: у кожного свій інтерпретатор і свій GIL, вони рахують на різних ядрах. На нашій 4-ядерній машині: послідовно ≈0,7 с, потоки ≈0,7–0,8 с, процеси ≈0,2 с.

</details>

### Приклад 3.2. Потоки проти процесів на обчисленнях

`ProcessPoolExecutor` має той самий інтерфейс, що й пул потоків: `map`, `submit`, `with`. Різниця — всередині: кожен працівник — окремий процес.

In [ ]:
timed("послідовно", lambda: [district_stats.count_primes(n) for n in DISTRICTS])
with ThreadPoolExecutor(max_workers=4) as pool:
    timed("4 потоки  ", lambda: list(pool.map(district_stats.count_primes, DISTRICTS)))
with ProcessPoolExecutor(max_workers=4) as pool:
    timed("4 процеси ", lambda: list(pool.map(district_stats.count_primes, DISTRICTS)))

### Приклад 3.3. Кожен працівник — окремий процес

`os.getpid()` — номер процесу в операційній системі. У ноутбука він один, у працівників пулу — свої:

In [ ]:
import os

print("процес ноутбука:", os.getpid())
with ProcessPoolExecutor(max_workers=3) as pool:
    pids = list(pool.map(district_stats.worker_pid, range(9)))
print("процеси-працівники:", sorted(set(pids)))
print("скільки задач зробив кожен:", {pid: pids.count(pid) for pid in sorted(set(pids))})

Дев'ять задач розійшлися між трьома процесами — і жоден із них не збігається з процесом ноутбука.

### Приклад 3.4. Навіщо `if __name__ == "__main__":`

У ноутбуці цієї перевірки немає — він і так головна програма. У `.py`-файлі код, що створює процеси, **обов'язково** стоїть під нею. Ось чому.

Python запускає процес-працівник одним із **способів старту**. Windows і macOS використовують `spawn`, Linux з Python 3.14 — `forkserver` (до 3.14 — `fork`). `spawn` і `forkserver` стартують чистий інтерпретатор, який **заново імпортує** твій головний файл, щоб знайти функцію. Якби пул створювався просто на рівні файлу, кожен імпорт створював би новий пул — і Python зупиняє це з `RuntimeError` про «bootstrapping phase». Код під `if __name__ == "__main__":` під час такого імпорту не виконується.

Ось як виглядає правильний файл `report.py`:

```python
from concurrent.futures import ProcessPoolExecutor

import district_stats


def main():
    with ProcessPoolExecutor() as pool:
        print(list(pool.map(district_stats.count_primes, [150_000] * 4)))


if __name__ == "__main__":      # працівники імпортують файл, але сюди не заходять
    main()
```

In [ ]:
import multiprocessing

print("Спосіб старту процесів тут:", multiprocessing.get_start_method())

### Приклад 3.5. Процеси не діляться пам'яттю

`district_stats.add_order` дописує замовлення в список `orders` модуля. Викличемо її в процесі-працівнику:

In [ ]:
district_stats.orders.clear()
with ProcessPoolExecutor(max_workers=1) as pool:
    print("що бачив працівник:", list(pool.map(district_stats.add_order, ["А", "Б", "В"])))
print("orders у ноутбуці:", district_stats.orders)

Працівник дописав три замовлення **у свою копію** списку (лічильник ріс 1, 2, 3). У ноутбуці список лишився порожнім: у процесів окрема пам'ять. Дані між процесами ходять лише **аргументами й результатами** — Python пакує їх через `pickle` і пересилає.

### Приклад 3.6. Не все можна передати в процес

Функція має бути **на верхньому рівні модуля**: lambda чи функцію, визначену всередині іншої функції, `pickle` не запакує. Відкритий файл чи з'єднання з базою теж не передаси — передавай шлях, а відкривай уже в працівнику.

In [ ]:
try:
    with ProcessPoolExecutor(max_workers=1) as pool:
        print(list(pool.map(lambda n: n * 2, [1, 2, 3])))
except Exception as error:
    print(type(error).__name__, "—", error)

**Процеси не безкоштовні.** Запустити працівника, запакувати й переслати дані — це час.

**🔮 Прогноз:** 4 крихітні задачі `count_primes(100)` — що швидше: послідовно чи в 4 процесах (разом зі стартом пулу)?

<details>
<summary>Відповідь</summary>

Послідовно — частки мілісекунди. Процеси — у сотні разів довше: майже весь час іде на старт працівників і пересилання даних. На Windows і macOS (`spawn`) різниця ще більша. Процеси окупаються, коли кожна задача рахує помітний час, а даних ходить мало. Тому — спершу виміряй.

</details>

### Приклад 3.7. Ціна процесів

In [ ]:
def in_processes(func, data):
    with ProcessPoolExecutor(max_workers=4) as pool:      # старт пулу теж рахуємо
        return list(pool.map(func, data))


def compare_small_tasks(data):
    start = time.perf_counter()
    [district_stats.count_primes(n) for n in data]
    sequential = time.perf_counter() - start

    start = time.perf_counter()
    in_processes(district_stats.count_primes, data)
    processes = time.perf_counter() - start

    print(f"послідовно: {sequential:.4f} с")
    print(f"у процесах: {processes:.4f} с — у {processes / sequential:.0f} разів повільніше")

In [ ]:
compare_small_tasks([100] * 4)

### Задача 2. `district_report(limits)`

Поверни **словник** «ліміт → кількість простих чисел» для кожного ліміту зі списку, рахуючи `district_stats.count_primes` **у процесах**.

Підказка: `pool.map` повертає результати в порядку входу — їх можна з'єднати з `limits` через `zip`.

In [ ]:
def district_report(limits):
    # BEGIN SOLUTION
    with ProcessPoolExecutor() as pool:
        return dict(zip(limits, pool.map(district_stats.count_primes, limits)))
    # END SOLUTION

In [ ]:
report = district_report([10, 100, 1000, 150_000])
print(report)
assert report == {10: 4, 100: 25, 1000: 168, 150_000: 13848}
print("✅ Задача 2 розв'язана")

### Приклад 3.8. Чекає **і** рахує: конвеєр «потоки → процеси»

Реальна задача часто змішана: спершу **завантажити** (чекати), потім **порахувати**. Кожен етап — своїм інструментом. Нехай «розмір» меню визначає, скільки рахувати:

In [ ]:
def download_menu(name):
    time.sleep(0.5)                       # мережа
    return 30_000 + len(name) * 2_000     # «розмір меню» → ліміт для підрахунку

In [ ]:
with ThreadPoolExecutor(max_workers=5) as threads:
    sizes = timed("1. завантажили 5 меню (потоки)  ", lambda: list(threads.map(download_menu, RESTAURANTS)))
with ProcessPoolExecutor(max_workers=4) as processes:
    stats = timed("2. порахували статистику (процеси)", lambda: list(processes.map(district_stats.count_primes, sizes)))
for name, count in zip(RESTAURANTS, stats):
    print(f"{name:14} {count}")

---
## 4. `asyncio`: тисячі очікувань в одному потоці

Потоки чекають одночасно, але кожен потік — це пам'ять і перемикання операційною системою. `asyncio` робить те саме **в одному потоці**: задачі самі кажуть «я зараз чекаю — працюй з іншими».

`async def` створює **корутину**: виклик не виконує тіло, а повертає об'єкт — як генераторна функція (урок 11). Запускає корутини **цикл подій** (event loop) — диспетчер, який виконує задачі по черзі в одному потоці й перемикається, коли задача чекає. Місце, де можна стати на паузу, позначає `await`.

У Jupyter і Colab цикл подій уже працює, тому **замість `asyncio.run(main())` пиши `await main()`** прямо в клітинці. У `.py`-файлі — навпаки, `asyncio.run(main())`: він створює цикл, виконує корутину й закриває цикл.

### Приклад 4.1. Корутина — це ще не виконання

In [ ]:
async def check_restaurant_async(name):
    await asyncio.sleep(0.5)          # чекаємо, але не блокуємо інших
    return f"{name}: відкрито"

In [ ]:
coro = check_restaurant_async("Борщ і Ко")
print(type(coro).__name__)            # тіло ще не виконувалось
print(await coro)                     # у файлі: asyncio.run(...)

Порівняй дві функції:

| | `def check_restaurant` | `async def check_restaurant_async` |
|---|---|---|
| Виклик | одразу виконує тіло | повертає корутину |
| Очікування | `time.sleep(0.5)` — **блокує** весь потік | `await asyncio.sleep(0.5)` — віддає керування циклу |
| Як отримати результат | просто результат виклику | `await корутина` |

**🔮 Прогноз:** що надрукує `kitchen()` і в якому порядку?

```python
async def cook(name, seconds):
    print(f"{name}: почав")
    await asyncio.sleep(seconds)
    print(f"{name}: закінчив")


async def kitchen():
    await asyncio.gather(cook("борщ", 0.2), cook("салат", 0.1))
```

<details>
<summary>Відповідь</summary>

`борщ: почав`, `салат: почав`, `салат: закінчив`, `борщ: закінчив`. Борщ доходить до `await` і стає на паузу — цикл подій запускає салат. Салат чекає менше, тому закінчує першим.

</details>

### Приклад 4.2. `gather` — запустити кілька корутин разом

In [ ]:
async def cook(name, seconds):
    print(f"{name}: почав")
    await asyncio.sleep(seconds)
    print(f"{name}: закінчив")


async def kitchen():
    await asyncio.gather(cook("борщ", 0.2), cook("салат", 0.1))

In [ ]:
await kitchen()

### Приклад 4.3. Шкала часу: де цикл перемикається

Додамо до кожного кроку час від старту. Стрілка → — корутина дійшла до `await` і віддала керування; ← — повернулася після очікування.

In [ ]:
async def check_traced(name, delay, start):
    print(f"{time.perf_counter() - start:.2f} с  → {name} чекає {delay} с")
    await asyncio.sleep(delay)
    print(f"{time.perf_counter() - start:.2f} с  ← {name} відповів")
    return name

In [ ]:
start = time.perf_counter()
order = await asyncio.gather(
    check_traced("Борщ і Ко", 0.3, start),
    check_traced("Піца Поділ", 0.1, start),
    check_traced("Суші Оболонь", 0.2, start),
)
print("gather повернув у порядку запуску:", order)

Усі троє стартували в момент 0,00, а повертались у порядку своїх затримок. Загальний час — найдовше очікування (0,3 с), а не сума (0,6 с). При цьому `gather` повертає результати **в порядку аргументів** — як `pool.map`.

### Приклад 4.4. `check_all_async(names)` — опитати всі ресторани

`gather(*корутини)` приймає корутини окремими аргументами, тому список розпаковуємо зірочкою.

In [ ]:
async def check_all_async(names):
    return await asyncio.gather(*(check_restaurant_async(n) for n in names))

In [ ]:
start = time.perf_counter()
statuses = await check_all_async(RESTAURANTS)
elapsed = time.perf_counter() - start
print(statuses)
print(f"{elapsed:.1f} с")
assert list(statuses) == [f"{n}: відкрито" for n in RESTAURANTS]
assert elapsed < 1.5

**🔮 Прогноз:** колега написав опитування з `await`, але без `gather`. Скільки воно займе?

```python
async def check_one_by_one(names):
    results = []
    for name in names:
        results.append(await check_restaurant_async(name))
    return results
```

<details>
<summary>Відповідь</summary>

2,5 с — як без `asyncio`. `await` чекає, поки **ця** корутина закінчиться, і лише потім цикл `for` бере наступний ресторан. Одночасно корутини йдуть, лише коли їх запущено разом: `gather` (або задачі `asyncio.create_task`).

</details>

### Приклад 4.5. Помилка: `await` у циклі

In [ ]:
async def check_one_by_one(names):
    results = []
    for name in names:
        results.append(await check_restaurant_async(name))
    return results

In [ ]:
start = time.perf_counter()
await check_one_by_one(RESTAURANTS)
print(f"await у циклі: {time.perf_counter() - start:.1f} с")

### Приклад 4.6. Помилка: блокуючий виклик у корутині

Інша поширена помилка — `time.sleep` (чи будь-який звичайний блокуючий виклик) усередині `async def`. Він зупиняє **весь** цикл подій: поки один ресторан «спить», ніхто інший не працює.

In [ ]:
async def check_blocking_inside(name):
    time.sleep(0.5)                       # ❌ блокує цикл подій
    return f"{name}: відкрито"


async def check_fixed(name):
    await asyncio.sleep(0.5)              # ✅ віддає керування циклу
    return f"{name}: відкрито"

In [ ]:
for check in (check_blocking_inside, check_fixed):
    start = time.perf_counter()
    await asyncio.gather(*(check(n) for n in RESTAURANTS))
    print(f"{check.__name__:22} {time.perf_counter() - start:.1f} с")

`gather` тут не допоміг: корутини запущені «разом», але `time.sleep` не віддає керування — вони однаково йдуть по черзі. Правило: в `async def` — лише `await`-версії очікування (`asyncio.sleep`, `aiohttp`, `httpx.AsyncClient`, асинхронні драйвери баз). Що робити з блокуючим кодом, який не переписати, — розділ 5.

### Приклад 4.7. Забута корутина

А якщо корутину створили й **забули** — без `await` і без `gather`? Її тіло так і не виконається, а Python, прибираючи об'єкт, попередить `RuntimeWarning`. Зловимо це попередження:

In [ ]:
import gc
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    check_restaurant_async("Піца Поділ")      # створили й забули: тіло не виконалось
    gc.collect()

for warning in caught:
    print(warning.category.__name__, "—", warning.message)

### Приклад 4.8. `create_task` — запустити й займатися іншим

`gather` чекає всіх одразу. `asyncio.create_task(корутина)` **запускає** корутину у фоні й одразу повертає задачу (`Task`). Поки вона чекає мережу, ми робимо іншу роботу, а результат забираємо потім через `await task`.

In [ ]:
async def morning():
    start = time.perf_counter()
    task = asyncio.create_task(check_restaurant_async("Борщ і Ко"))   # пішла у фон
    print(f"{time.perf_counter() - start:.2f} с  задачу запущено, готуємо звіт…")

    await asyncio.sleep(0.3)                                           # інша робота
    print(f"{time.perf_counter() - start:.2f} с  звіт готовий, чекаємо ресторан")

    result = await task
    print(f"{time.perf_counter() - start:.2f} с  {result}")

In [ ]:
await morning()

Разом ≈0,5 с, а не 0,3 + 0,5 = 0,8: ресторан «чекав» паралельно зі звітом.

### Приклад 4.9. `asyncio.as_completed` — у порядку готовності

Як і в потоках (приклад 1.5), можна обробляти результати щойно вони готові:

In [ ]:
async def check_delay(name, delay):
    await asyncio.sleep(delay)
    return f"{name} (відповів за {delay} с)"

In [ ]:
start = time.perf_counter()
coros = [check_delay(name, delay) for name, delay in DELAYS.items()]
for next_done in asyncio.as_completed(coros):
    result = await next_done                 # чекаємо найближчу готову
    print(f"{time.perf_counter() - start:.1f} с  {result}")

### Приклад 4.10. Помилки в `gather`

За замовчуванням перший виняток у будь-якій корутині вилітає з `gather` — і результатів решти ти не побачиш. `return_exceptions=True` повертає винятки **як значення** в списку результатів — поруч з успішними.

In [ ]:
async def check_maybe_closed_async(name):
    await asyncio.sleep(0.1)
    if name == "Суші Оболонь":
        raise ConnectionError(f"{name}: сервер не відповідає")
    return f"{name}: відкрито"

In [ ]:
try:
    await asyncio.gather(*(check_maybe_closed_async(n) for n in RESTAURANTS))
except ConnectionError as error:
    print("без return_exceptions:", error)

print()
results = await asyncio.gather(*(check_maybe_closed_async(n) for n in RESTAURANTS),
                               return_exceptions=True)
for result in results:
    label = "помилка" if isinstance(result, Exception) else "ok     "
    print(label, result)

### Приклад 4.11. `TaskGroup` (Python 3.11+)

`asyncio.TaskGroup` — сучасна заміна `gather`: задачі створюються в блоці `async with`, і вихід з блоку чекає **всіх**. Якщо одна задача впала, група **скасовує решту** й кидає `ExceptionGroup` з усіма помилками. Це безпечніше: жодна задача не лишиться «висіти» у фоні.

Курс підтримує Python 3.10+, тому приклад перевіряє версію.

In [ ]:
async def morning_group(names, check):
    async with asyncio.TaskGroup() as group:
        tasks = [group.create_task(check(name)) for name in names]
    return [task.result() for task in tasks]      # тут усі задачі вже завершені

In [ ]:
if sys.version_info >= (3, 11):
    print(await morning_group(RESTAURANTS, check_restaurant_async))
    try:
        await morning_group(RESTAURANTS, check_maybe_closed_async)
    except ExceptionGroup as group_error:
        print("група скасувала решту, помилки:", [str(e) for e in group_error.exceptions])
else:
    print("TaskGroup з'явився в Python 3.11 — тут", sys.version.split()[0])

### Приклад 4.12. Тайм-аут: ресторан, що мовчить

Ресторан, що не відповів за `timeout` секунд, отримує статус `"<назва>: не відповідає"`. `asyncio.wait_for(корутина, timeout=...)` скасовує корутину після тайм-ауту й кидає `asyncio.TimeoutError` (на Python 3.10 це не той самий клас, що вбудований `TimeoutError`; з 3.11 — той самий).

In [ ]:
async def check_with_delay(name, delay):
    await asyncio.sleep(delay)
    return f"{name}: відкрито"


async def safe_check(name, delay, timeout=1.0):
    try:
        return await asyncio.wait_for(check_with_delay(name, delay), timeout=timeout)
    except asyncio.TimeoutError:
        return f"{name}: не відповідає"

In [ ]:
delays = {"Борщ і Ко": 3.0, "Піца Поділ": 0.5, "Суші Оболонь": 0.5, "Вареники 24/7": 0.5, "Шаурма Центр": 0.5}
start = time.perf_counter()
lines = await asyncio.gather(*(safe_check(n, d) for n, d in delays.items()))
elapsed = time.perf_counter() - start
for line in lines:
    print(line)
print(f"{elapsed:.1f} с")
assert lines[0] == "Борщ і Ко: не відповідає"
assert all(line.endswith("відкрито") for line in lines[1:])
assert elapsed < 2.0

Разом ≈1 с, а не 3: «Борщ і Ко» скасовано після тайм-ауту, решта встигли. З Python 3.11 те саме можна записати блоком `async with asyncio.timeout(1.0):`.

### Приклад 4.13. Черга `asyncio.Queue`: кур'єри без потоків

Той самий патерн «виробник — споживач», що в прикладі 1.7, але в одному потоці. `await queue.get()` чекає замовлення, не блокуючи цикл; `queue.task_done()` + `await queue.join()` — «дочекатися, поки все розвезуть».

In [ ]:
async def courier_async(name, orders, delivered):
    while True:
        order = await orders.get()
        await asyncio.sleep(0.2)                 # везе
        delivered.append((name, order))
        orders.task_done()                       # позначили замовлення виконаним


async def shift(order_list, courier_names):
    orders = asyncio.Queue()
    delivered = []
    for order in order_list:
        orders.put_nowait(order)
    workers = [asyncio.create_task(courier_async(n, orders, delivered)) for n in courier_names]
    await orders.join()                          # чекаємо, поки черга спорожніє
    for worker in workers:
        worker.cancel()                          # кур'єри йдуть додому
    return delivered

In [ ]:
start = time.perf_counter()
done = await shift(["борщ", "піца", "суші", "вареники", "шаурма", "узвар"], ["Тарас", "Марія"])
print(f"{len(done)} доставок за {time.perf_counter() - start:.1f} с")
for name in ("Тарас", "Марія"):
    print(name, "→", [order for who, order in done if who == name])

---
## 5. Блокуючий код, який не можна переписати

У прикладі 4.6 помилка виправлялась легко: `time.sleep` → `await asyncio.sleep`. Але буває, що блокуючу функцію переписати не можна: стара бібліотека, `requests.get`, синхронний драйвер бази. Такий виклик в `async def` зупинить увесь цикл подій.

Вихід — `await asyncio.to_thread(функція, аргументи...)`: функція виконується в окремому потоці, а корутина чекає її, не блокуючи цикл.

### Приклад 5.1. `check_all_blocking(names)` через `to_thread`

Опитуємо ресторани **звичайною** `check_restaurant` (вона блокує, її не чіпаємо), але одночасно:

In [ ]:
async def check_all_blocking(names):
    return await asyncio.gather(*(asyncio.to_thread(check_restaurant, n) for n in names))

In [ ]:
start = time.perf_counter()
statuses = await check_all_blocking(RESTAURANTS)
elapsed = time.perf_counter() - start
print(statuses[0], f"| {elapsed:.1f} с")
assert list(statuses) == [f"{n}: відкрито" for n in RESTAURANTS]
assert elapsed < 1.5

### Приклад 5.2. Обчислення — у процеси навіть з `asyncio`

`to_thread` бере потоки зі стандартного пулу циклу подій (`min(32, ядра + 4)`), тож на тисячах викликів час упреться в цей розмір. А для **обчислень** на чистому Python `to_thread` не допоможе — GIL той самий. Їх відправляють у пул процесів через `loop.run_in_executor`:

In [ ]:
async def district_stats_async(limits):
    loop = asyncio.get_running_loop()
    with ProcessPoolExecutor(max_workers=4) as pool:
        jobs = [loop.run_in_executor(pool, district_stats.count_primes, n) for n in limits]
        return await asyncio.gather(*jobs)

In [ ]:
start = time.perf_counter()
print(await district_stats_async(DISTRICTS))
print(f"{time.perf_counter() - start:.1f} с — цикл подій тим часом був вільний")

---
## 6. Обмежити одночасність: `asyncio.Semaphore`

`gather` запускає **все одразу**: 5 ресторанів — 5 запитів, 5000 — 5000. Сервер може такого не витримати або заблокувати тебе. У пулі потоків межу задає `max_workers`, в `asyncio` — **семафор**: `asyncio.Semaphore(2)` пускає в захищений блок `async with semaphore:` не більше двох корутин одночасно, решта чекають біля входу.

**🔮 Прогноз:** 5 ресторанів по 0,5 с, межа — 2. Скільки часу?

<details>
<summary>Відповідь</summary>

Близько 1,5 с: три «хвилі» — 2 + 2 + 1.

</details>

### Приклад 6.1. Лічильник «у польоті»

`check_counted` сама рахує, скільки запитів виконується одночасно, і запам'ятовує максимум. Спершу — **без** семафора:

In [ ]:
in_flight = 0
max_in_flight = 0


async def check_counted(name):
    global in_flight, max_in_flight
    in_flight += 1
    max_in_flight = max(max_in_flight, in_flight)
    await asyncio.sleep(0.5)
    in_flight -= 1
    return f"{name}: відкрито"

In [ ]:
max_in_flight = 0
await asyncio.gather(*(check_counted(n) for n in RESTAURANTS))
print("без семафора одночасно:", max_in_flight)

Усі п'ять одночасно. (Тут `global` у корутинах безпечний: `asyncio` перемикається **лише на `await`**, а між `in_flight += 1` і `max(...)` немає жодного `await`. У потоках так не можна — розділ 2.)

### Задача 3. `check_limited(names, limit)`

Допиши `one(name)`: виклик `check_counted(name)` має бути всередині `async with semaphore:`. Перевірка — у наступній клітинці.

In [ ]:
async def check_limited(names, limit):
    semaphore = asyncio.Semaphore(limit)

    async def one(name):
        # BEGIN SOLUTION
        async with semaphore:
            return await check_counted(name)
        # END SOLUTION

    return await asyncio.gather(*(one(n) for n in names))

In [ ]:
max_in_flight = 0
start = time.perf_counter()
statuses = await check_limited(RESTAURANTS, limit=2)
elapsed = time.perf_counter() - start
print(f"одночасно щонайбільше: {max_in_flight} | {elapsed:.1f} с")
assert list(statuses) == [f"{n}: відкрито" for n in RESTAURANTS]
assert max_in_flight == 2, "одночасно мають працювати рівно 2 запити"
assert 1.3 < elapsed < 2.2
print("✅ Задача 3 розв'язана")

Семафор обмежує, скільки запитів **у польоті одночасно**, а не скільки їх **за секунду**. Ліміти API («не більше 10 запитів за секунду») — окрема річ. Якщо сервер відповів `429 Too Many Requests`, він часто додає заголовок `Retry-After` — скільки секунд почекати. Тоді чекають і повторюють, щоразу довше (backoff). Детальніше — у книзі й в уроці 32.

---
## 7. Одна задача — чотири способи

### Приклад 7.1. Опитати 5 ресторанів: порівняння

Зведемо разом усе, що робили з ресторанами. Задача та сама — інструменти різні.

In [ ]:
def run_sequential():
    return [check_restaurant(n) for n in RESTAURANTS]


def run_threads():
    with ThreadPoolExecutor(max_workers=5) as pool:
        return list(pool.map(check_restaurant, RESTAURANTS))


async def run_asyncio():
    return await asyncio.gather(*(check_restaurant_async(n) for n in RESTAURANTS))


async def run_to_thread():
    return await asyncio.gather(*(asyncio.to_thread(check_restaurant, n) for n in RESTAURANTS))

In [ ]:
timed("послідовно            ", run_sequential)
timed("потоки                ", run_threads)

for label, run in (("asyncio               ", run_asyncio), ("asyncio + to_thread   ", run_to_thread)):
    start = time.perf_counter()
    await run()
    print(f"{label}: {time.perf_counter() - start:.1f} с")

Для задачі, що **чекає**, потоки й `asyncio` дають однаковий виграш. Різниця — у масштабі й у тому, який код у тебе вже є.

## Що обрати

| Робота | З чого почати |
|---|---|
| Чекає, десятки задач, звичайний (синхронний) код | `ThreadPoolExecutor` |
| Чекає, багато одночасних з'єднань, асинхронні бібліотеки | `asyncio` |
| Чекає, але бібліотека блокуюча, а решта коду асинхронна | `asyncio.to_thread` |
| Рахує на чистому Python, GIL увімкнено | `ProcessPoolExecutor` — після виміру накладних витрат |
| Чекає **і** рахує | завантажити потоками чи `asyncio`, потім рахувати в пулі процесів (приклад 3.8) |
| Спільні дані між потоками | спершу — не ділити (приклад 2.4); якщо треба — `Lock` на мінімальний блок |

Обирай найпростіший інструмент, що підходить, **обмежуй** одночасність і **міряй**.

---
## Самоперевірка

1. Чим задача, що чекає, відрізняється від задачі, що рахує? Що прискорює кожну?
2. Чому потоки не прискорили `count_primes`?
3. Як уникнути стану гонитви без `Lock`?
4. Чому `district_stats.orders` у ноутбуці лишився порожнім?
5. Чим `asyncio.run(main())` у файлі відрізняється від `await main()` у ноутбуці?
6. Чому перевірку `if name not in imported` і `append` треба тримати під одним замком, а завантаження — поза ним?
7. Коли пул процесів повільніший за звичайний цикл? Чому lambda не можна передати в `ProcessPoolExecutor`?
8. Чому `await` у циклі `for` не пришвидшує опитування?
9. Як викликати блокуючий `requests.get` з асинхронного коду, не зупиняючи цикл подій?
10. Що обмежує `asyncio.Semaphore(3)`, а чого — ні?
11. Чим `pool.map` відрізняється від `submit` + `as_completed`?
12. Які чотири умови потрібні для взаємного блокування і як найпростіше його уникнути?
13. Що станеться з рештою задач, якщо одна впаде в `gather` без `return_exceptions` і в `TaskGroup`?

<details>
<summary>Відповіді</summary>

1. Задача, що чекає, здебільшого простоює (мережа, диск) — допомагають потоки або `asyncio`. Задача, що рахує, займає процесор — допомагають процеси.
2. Через GIL: у кожен момент байт-код виконує лише один потік програми.
3. Не ділити змінних: кожен потік повертає результат, головна програма їх збирає.
4. Процес-працівник має власну копію пам'яті; дані повертаються лише результатами функцій.
5. `asyncio.run` створює цикл подій; у ноутбуці цикл уже працює, тому корутину просто чекають через `await`.
6. Інакше два потоки обидва побачать «ще немає» і обидва допишуть (check-then-act). Завантаження під замком пішли б по черзі — потоки втратили б сенс.
7. Коли задачі дрібні: старт працівників і пересилання даних дорожчі за саму роботу. Аргументи й функції пакуються через `pickle`, а lambda він не пакує — потрібна функція верхнього рівня модуля.
8. `await` чекає, поки закінчиться ця корутина, і лише потім цикл бере наступну. Одночасно корутини йдуть через `gather` чи `create_task`.
9. `await asyncio.to_thread(requests.get, url)` — виклик іде в окремому потоці.
10. Скільки корутин одночасно в захищеному блоці. Не обмежує кількість запитів за секунду — це окремо: паузи, `Retry-After`, backoff.
11. `map` повертає результати в порядку входу; `as_completed` — у порядку готовності, а `submit` дозволяє різні аргументи й обробку помилок кожної задачі окремо.
12. Ресурс у одного власника; тримає один — чекає інший; ресурс не відбирають; цикл очікування. Найпростіше — брати замки завжди в одному порядку.
13. `gather` кидає перший виняток, але інші корутини продовжують працювати у фоні; `TaskGroup` скасовує решту задач і кидає `ExceptionGroup` з усіма помилками.

</details>

## Далі

- Теорія, покрокові схеми гонитви й циклу подій, вибір інструменту: [Урок 28](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_28/).
- Документація: [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html), [`threading`](https://docs.python.org/3/library/threading.html), [`multiprocessing`](https://docs.python.org/3/library/multiprocessing.html), [`asyncio` — Coroutines and Tasks](https://docs.python.org/3/library/asyncio-task.html) (розділи «Coroutines», «Awaitables», «Running Tasks Concurrently», «Running in Threads»).
- Почитати після уроку: [Real Python — Speed Up Your Python Program With Concurrency](https://realpython.com/python-concurrency/) (≈40 хв), [SuperFastPython — як обрати API для конкурентності](https://superfastpython.com/python-concurrency-choose-api/).
- Без GIL: [Python support for free threading](https://docs.python.org/3/howto/free-threading-python.html), [PEP 779](https://peps.python.org/pep-0779/) — чому з 3.14 збірка без GIL офіційно підтримується.
- **Поглиблений рівень — бонусні уроки модуля 4** (`module_4/bonus/`). Там ці самі теми розписано значно детальніше, на реальних проєктах: [потоки й конкурентність](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_threading_concurrency/note_lesson_32_threading_concurrency.ipynb) і [багатопотоковий скрейпер](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_threading_concurrency/note_lesson_32_multithreaded_scraper.ipynb); [multiprocessing](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_multiprocessing/note_lesson_33_multiprocessing.ipynb) і [лабораторна: паралельний конвеєр обробки даних](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_multiprocessing/lab_lesson_33_flood_pipeline.ipynb); [asyncio](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_asyncio/lesson_34_asyncio.ipynb) з асинхронним FastAPI-демо й новинним дашбордом. Перед ними корисні [мережа: сокети й TCP](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_networking_base/note_lesson_30_networking_base.ipynb) і [HTTP-запити](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/bonus/lesson_http_requests/note_lesson_31_http_requests.ipynb).
- **Урок 29** — практикум П6: стек, черга, купа, префіксне дерево й LRU-кеш.